In [31]:
# ============================================================
# RESCUE DRONE RL ENVIRONMENT
#
# Custom Gymnasium Environment
#
# State Space:
#   Position (6x6 Grid)
#   Battery Level
#   Rescue Status of Targets
#
# Action Space:
#   Move Up
#   Move Down
#   Move Left
#   Move Right
#   Hover
#
# ============================================================

In [32]:
# IMPORTS
import gymnasium as gym
from gymnasium import spaces
from gymnasium.envs.registration import register

import numpy as np
import random
import time
from io import BytesIO
from PIL import Image

from collections import namedtuple
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython.display import display
import ipywidgets as widgets

In [33]:
# ============================================================
# REGISTER ENVIRONMENT (Will be done after class definition)
# ============================================================

# Note: Environment registration happens after RescueDroneEnv is defined

In [34]:
# ============================================================
# ACTION DEFINITIONS AND RESCUE STATUS
# ============================================================

ACTION_SELECTION = namedtuple("ActionSelection",\
                              ["MOVE_UP","MOVE_DOWN",\
                               "MOVE_LEFT","MOVE_RIGHT","HOVER"])

RESCUE_STATUS = namedtuple("RescueStatus",\
                           ["NOT_RESCUED",\
                            "RESCUED"])
        
a_sel = ACTION_SELECTION(0, 1, 2, 3, 4)
r_status = RESCUE_STATUS(0, 1)

In [ ]:
# ============================================================
# 1. Custom Drone Rescue Environment 
# ============================================================

class RescueDroneEnv(gym.Env):

    metadata = {
        "render_modes": ["human"],
        "render_fps": 5
    }

    def __init__(self, render_mode="human"):

        super().__init__()

        # Environment Configuration
        self.grid_rows = 6
        self.grid_cols = 6
        self.max_steps = 75
        self.MAX_BATTERY = 10
        self.START_BATTERY = 4
        self.render_mode = render_mode

        # Fixed Environment Layout
        self.start_position = (0, 0)
        self.charging_stations = [(0, 1), (4, 0)]
        self.danger_zones = [(1, 1), (2, 4), (4, 2), (5, 5)]
        self.wind_zones = [(0, 3), (3, 3)]
        self.blocked_cells = [(2, 2), (3, 1), (4, 4)]
        self.targets = [(2, 0), (2, 1), (5, 2)]

        # Action Space
        self.action_space = spaces.Discrete(5)

        # Observation Space
        self.observation_space = spaces.Dict({
            "position": spaces.MultiDiscrete([self.grid_rows, self.grid_cols]),
            "battery": spaces.Discrete(self.MAX_BATTERY + 1),
            "rescued": spaces.MultiBinary(3)
        })

        # Runtime Variables
        self.position = list(self.start_position)
        self.battery_level = self.START_BATTERY
        self.rescued_targets = [r_status.NOT_RESCUED, r_status.NOT_RESCUED, r_status.NOT_RESCUED]
        self.step_count = 0

        # Render Frames
        self.render_frames = []

        # Rendering Configuration
        self.window_size = 720
        self.cell_size = self.window_size // self.grid_rows
        plt.close('all')

    def get_observation(self):
        '''Create observation returned to the agent.'''
        return {
            "position": np.array(self.position, dtype=np.int32),
            "battery": self.battery_level,
            "rescued": np.array(self.rescued_targets, dtype=np.int8)
        }

    def is_charging_station(self, position: tuple) -> bool:
        return position in self.charging_stations

    def is_danger_zone(self, position: tuple) -> bool:
        return position in self.danger_zones

    def is_wind_zone(self, position: tuple) -> bool:
        return position in self.wind_zones

    def is_blocked_cell(self, position: tuple) -> bool:
        return position in self.blocked_cells

    def is_target(self, position: tuple) -> bool:
        return position in self.targets

    def recharge_battery(self):
        self.battery_level = self.MAX_BATTERY

    def update_rescue_status(self):
        '''Update rescue status if a target is reached.'''
        current_position = tuple(self.position)
        for idx, target in enumerate(self.targets):
            if (current_position == target and 
                self.rescued_targets[idx] == r_status.NOT_RESCUED):
                self.rescued_targets[idx] = r_status.RESCUED

    def all_targets_rescued(self):
        return all(status == r_status.RESCUED for status in self.rescued_targets)

    def apply_wind_disturbance(self, action: int) -> int:
        '''Wind cell introduces stochastic action. 30% chance of random movement.'''
        current_position = tuple(self.position)
        if not self.is_wind_zone(current_position):
            return action
        if action == a_sel.HOVER:
            return action
        if np.random.random() < 0.30:
            action = random.choice([a_sel.MOVE_UP, a_sel.MOVE_DOWN, a_sel.MOVE_LEFT, a_sel.MOVE_RIGHT])
        return action

    def get_next_position(self, action: int) -> tuple:
        '''Compute next position after action execution.'''
        row, col = self.position
        if action == a_sel.MOVE_UP:
            row -= 1
        elif action == a_sel.MOVE_DOWN:
            row += 1
        elif action == a_sel.MOVE_LEFT:
            col -= 1
        elif action == a_sel.MOVE_RIGHT:
            col += 1
        elif action == a_sel.HOVER:
            pass
        row = max(0, min(row, self.grid_rows - 1))
        col = max(0, min(col, self.grid_cols - 1))
        return (row, col)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.position = list(self.start_position)
        self.battery_level = self.START_BATTERY
        self.rescued_targets = [r_status.NOT_RESCUED, r_status.NOT_RESCUED, r_status.NOT_RESCUED]
        self.step_count = 0
        observation = self.get_observation()
        info = {}
        if self.render_mode == "human":
            self.render()
        self.render_frames.clear()
        self.render_frames.append(self.render_to_frame())
        return (observation, info)

    def get_reward(self, previous_rescue_status) -> int:
        '''Calculate reward for the current state.'''
        reward = -1
        current_position = tuple(self.position)
        
        # New Rescue Achieved
        for idx in range(len(self.rescued_targets)):
            if (previous_rescue_status[idx] == r_status.NOT_RESCUED and
                self.rescued_targets[idx] == r_status.RESCUED):
                reward = 20
                return reward
        
        # Charging Station
        if self.is_charging_station(current_position):
            reward = 5
        
        # Danger Zone
        if self.is_danger_zone(current_position):
            reward = -10
        
        return reward

    def check_termination(self) -> bool:
        '''Episode terminates if: 1) Battery empty 2) All targets rescued'''
        battery_empty = self.battery_level <= 0
        all_rescued = self.all_targets_rescued()
        return battery_empty or all_rescued

    def step(self, action: int):
        '''Execute one environment step.'''
        self.step_count += 1
        previous_rescue_status = self.rescued_targets.copy()
        action = self.apply_wind_disturbance(action)
        self.battery_level -= 1
        candidate_position = self.get_next_position(action)
        
        if not self.is_blocked_cell(candidate_position):
            self.position = list(candidate_position)
        
        if self.is_charging_station(tuple(self.position)):
            self.recharge_battery()
        
        self.update_rescue_status()
        reward = self.get_reward(previous_rescue_status)
        
        if self.battery_level <= 0:
            reward += -20
        
        terminated = self.check_termination()
        truncated = self.step_count >= self.max_steps
        observation = self.get_observation()
        info = {
            "battery": self.battery_level,
            "step_count": self.step_count,
            "rescued_targets": self.rescued_targets
        }
        
        if self.render_mode == "human":
            self.render()
        self.render_frames.append(self.render_to_frame())
        
        return (observation, reward, terminated, truncated, info)

    def render_to_frame(self):
        '''Render current state to numpy array with colored cells and letter labels.'''
        from PIL import ImageDraw, ImageFont
        
        cell_pixel_size = 120
        grid_size = self.grid_rows * cell_pixel_size
        
        # Create image with white background
        img = Image.new('RGB', (grid_size, grid_size), color='white')
        draw = ImageDraw.Draw(img)
        
        # Color map for special cells
        color_map = {
            'free': 'white',
            'charging': '#00C800',      # Green
            'danger': '#DC0000',        # Red
            'wind': '#FFFF00',          # Yellow
            'blocked': '#000000',       # Black
            'target_unrescued': '#0064FF',  # Blue
            'start': '#B4B4B4'          # Gray
        }
        
        # First pass: Draw colored background rectangles
        for row in range(self.grid_rows):
            for col in range(self.grid_cols):
                x0 = col * cell_pixel_size
                y0 = row * cell_pixel_size
                x1 = x0 + cell_pixel_size
                y1 = y0 + cell_pixel_size
                
                cell_position = (row, col)
                cell_type = 'free'
                
                # Determine cell type
                if cell_position in self.charging_stations:
                    cell_type = 'charging'
                elif cell_position in self.danger_zones:
                    cell_type = 'danger'
                elif cell_position in self.wind_zones:
                    cell_type = 'wind'
                elif cell_position in self.blocked_cells:
                    cell_type = 'blocked'
                elif cell_position in self.targets:
                    target_index = self.targets.index(cell_position)
                    if self.rescued_targets[target_index] == r_status.NOT_RESCUED:
                        cell_type = 'target_unrescued'
                    else:
                        cell_type = 'free'
                
                if cell_position == self.start_position and cell_type == 'free':
                    cell_type = 'start'
                
                # Draw cell background
                draw.rectangle([x0, y0, x1, y1], fill=color_map[cell_type])
        
        # Draw grid lines
        for row in range(self.grid_rows + 1):
            y = row * cell_pixel_size
            draw.line([(0, y), (grid_size, y)], fill='#323232', width=2)
        
        for col in range(self.grid_cols + 1):
            x = col * cell_pixel_size
            draw.line([(x, 0), (x, grid_size)], fill='#323232', width=2)
        
        # Create fonts
        try:
            label_font = ImageFont.load_default()
            drone_font = ImageFont.load_default()
        except:
            label_font = None
            drone_font = None
        
        # Second pass: Draw cell labels
        for row in range(self.grid_rows):
            for col in range(self.grid_cols):
                x0 = col * cell_pixel_size
                y0 = row * cell_pixel_size
                x_center = x0 + cell_pixel_size // 2
                y_center = y0 + cell_pixel_size // 2
                
                cell_position = (row, col)
                text_label = ''
                
                # Determine label
                if cell_position in self.charging_stations:
                    text_label = 'C'
                elif cell_position in self.danger_zones:
                    text_label = 'X'
                elif cell_position in self.wind_zones:
                    text_label = 'W'
                elif cell_position in self.blocked_cells:
                    text_label = 'X'
                elif cell_position in self.targets:
                    target_index = self.targets.index(cell_position)
                    if self.rescued_targets[target_index] == r_status.NOT_RESCUED:
                        text_label = 'R'
                
                if cell_position == self.start_position:
                    if text_label == '':
                        text_label = 'S'
                
                # Draw label text
                if text_label:
                    draw.text((x_center, y_center), text_label, fill='white', anchor='mm', font=label_font)
        
        # Draw drone label with larger black font
        drone_row, drone_col = self.position
        drone_x = drone_col * cell_pixel_size + cell_pixel_size // 2
        drone_y = drone_row * cell_pixel_size + cell_pixel_size // 2
        
        # Draw drone with large bold black font (multiple overlapping offsets for bold effect)
        for offset_x in range(-3, 4):
            for offset_y in range(-3, 4):
                #make the drone white with black outline for better visibility and no fill and bigger font size and bold for better visibility.
                draw.text((drone_x + offset_x, drone_y + offset_y), 'D', fill='black', anchor='mm', font=drone_font)
                draw.text((drone_x + offset_x, drone_y + offset_y), 'D', fill='white', anchor='mm', font=drone_font)
        
        # Create final image with status text at bottom
        status_text = f'Battery: {self.battery_level} | Step: {self.step_count}'
        final_img = Image.new('RGB', (grid_size, grid_size + 50), color='wheat')
        final_img.paste(img, (0, 0))
        draw_final = ImageDraw.Draw(final_img)
        draw_final.text((grid_size // 2, grid_size + 25), status_text, fill='black', anchor='mm')
        
        return np.array(final_img)

    def render(self):
        '''Capture frame for notebook playback.'''
        frame = self.render_to_frame()
        self.render_frames.append(frame.copy())

    def playback_rendering(self):
        '''Display captured frames with interactive slider control.'''
        if len(self.render_frames) == 0:
            print("No frames captured. Run an episode first.")
            return
        
        image_widget = widgets.Image()
        slider = widgets.IntSlider(value=0, min=0, max=len(self.render_frames) - 1, 
                                   step=1, description="Frame:")
        
        def update(change):
            '''Update image when slider changes'''
            frame = self.render_frames[slider.value]
            buffer = BytesIO()
            Image.fromarray(frame).save(buffer, format="PNG")
            image_widget.value = buffer.getvalue()
        
        slider.observe(update, names="value")
        update(None)
        
        display(widgets.VBox([slider, image_widget]))

In [36]:
if __name__ == "__main__":

    print("RESCUE DRONE ENVIRONMENT CREATED")

    # Create environment directly from the notebook class
    env = RescueDroneEnv(render_mode="human")

    observation, info = env.reset()

    print(f"Initial Observation: {observation}")

    episode_over = False
    total_reward = 0

    while not episode_over:

        action = env.action_space.sample()

        observation, reward, terminated, truncated, info = env.step(action)

        total_reward += reward

        if terminated:
            print("Episode Terminated")

        if truncated:
            print("Episode Truncated")

        episode_over = (terminated or truncated)

        time.sleep(0.5)

    print(f"Total Reward: {total_reward}")
    print(f"Final Observation: {observation}")
    
    # Display interactive playback
    print("\nGenerating playback visualization...")
    env.playback_rendering()

RESCUE DRONE ENVIRONMENT CREATED
Initial Observation: {'position': array([0, 0], dtype=int32), 'battery': 4, 'rescued': array([0, 0, 0], dtype=int8)}
Episode Terminated
Total Reward: 18
Final Observation: {'position': array([2, 1], dtype=int32), 'battery': 0, 'rescued': array([1, 1, 0], dtype=int8)}

Generating playback visualization...
